In [15]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torchvision
from PIL import Image

train_data = np.load('Numarul4-train/1_4_train.npz', allow_pickle=True)
x_train, y_train = train_data['x'], train_data['y']
x_train = x_train.astype(np.float32)
print(x_train.shape, x_train.dtype, y_train.shape, y_train.dtype)

test_data = np.load('Numarul4-test/1_4_test.npz', allow_pickle=True)
x_test, y_test = test_data['x'], test_data['y']
x_test = x_test.astype(np.float32)
print(x_test.shape, x_test.dtype, y_test.shape, y_test.dtype)

x_train = torch.as_tensor(x_train)
x_test = torch.as_tensor(x_test)

N_CLASSES = len(np.unique(y_train))
print(f"N_CLASSES: {N_CLASSES}")

mean = torch.mean(input=x_train, dim=0)
std = torch.std(input=x_train, dim=0)
x_train_norm = (x_train - mean) / std
x_test_norm = (x_test - mean) / std

n = x_train_norm.shape[0]
K = (x_train_norm.T @ x_train_norm) / (n - 1)
print(K)

D, V = torch.linalg.eigh(K)
idx = torch.argsort(D, descending=True)

D = D[idx]
V = V[:, idx]

FEATURES = 400
x_train_PCA = x_train_norm @ V[:, :FEATURES]
x_test_PCA  = x_test_norm @ V[:, :FEATURES] 


(32801, 1024) float32 (32801,) int64
(11849, 1024) float32 (11849,) int64
N_CLASSES: 5
tensor([[1.0000, 0.9911, 0.9673,  ..., 0.5504, 0.5370, 0.5277],
        [0.9911, 1.0000, 0.9891,  ..., 0.5604, 0.5456, 0.5352],
        [0.9673, 0.9891, 1.0000,  ..., 0.5733, 0.5569, 0.5451],
        ...,
        [0.5504, 0.5604, 0.5733,  ..., 1.0000, 0.9846, 0.9544],
        [0.5370, 0.5456, 0.5569,  ..., 0.9846, 1.0000, 0.9878],
        [0.5277, 0.5352, 0.5451,  ..., 0.9544, 0.9878, 1.0000]])


In [ ]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.feature_vectors = x
        self.labels = y

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feature_vector = self.feature_vectors[idx]
        label = self.labels[idx]

        feature_vector = torch.as_tensor(data=feature_vector, dtype=torch.float32)
        label = torch.as_tensor(data=label, dtype=torch.int64)

        return feature_vector, label

train_dataset = CustomDataset(x_train_PCA, y_train)
test_dataset = CustomDataset(x_test_PCA, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False) 

torch.Size([32801, 1024])
False False
[0 1 2 3 4]


In [ ]:
class Model(nn.Module):
    def __init__(self, in_features=FEATURES, n_classes=N_CLASSES, act=nn.functional.relu):
        super().__init__()
        self.act = act

        self.fc1 = nn.Linear(in_features=in_features, out_features=300)
        self.drop1 = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(in_features=300, out_features=256)
        self.drop2 = nn.Dropout(p=0.4)
        self.fc3 = nn.Linear(in_features=256, out_features=64)
        self.drop3 = nn.Dropout(p=0.5)
        self.fc4 = nn.Linear(in_features=64, out_features=n_classes)


    def forward(self, x):
        x = self.drop1(self.act(self.fc1(x)))
        x = self.drop2(self.act(self.fc2(x)))
        x = self.drop3(self.act(self.fc3(x)))
        x = self.fc4(x)

        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
epochs = 15

def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def test(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            total_loss += loss.item()
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return total_loss / len(loader), correct / total

best = 0
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = test(model, test_loader, loss_fn, device)
    print(f"train_loss: {train_loss:.4f} || train_acc: {train_acc:.4f} || test_loss: {test_loss:.4f} || test_acc: {test_acc:.4f}")
    if best < test_acc:
        best = test_acc
        torch.save(model, 'model.pt')

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return correct / total


model = torch.load('model.pt', weights_only=False)
test_acc = evaluate(model, test_loader, device)
print(f'{test_acc:.4f}')

                                  # etichete care încep de la 0

train_loss: 0.9365 || train_acc: 0.6399 || test_loss: 0.5365 || test_acc: 0.8228
train_loss: 0.5134 || train_acc: 0.8303 || test_loss: 0.4458 || test_acc: 0.8524
train_loss: 0.4251 || train_acc: 0.8624 || test_loss: 0.4207 || test_acc: 0.8600
train_loss: 0.3736 || train_acc: 0.8777 || test_loss: 0.4018 || test_acc: 0.8709
train_loss: 0.3387 || train_acc: 0.8898 || test_loss: 0.3993 || test_acc: 0.8746
train_loss: 0.3117 || train_acc: 0.8971 || test_loss: 0.3737 || test_acc: 0.8795
train_loss: 0.2952 || train_acc: 0.9045 || test_loss: 0.3854 || test_acc: 0.8791
train_loss: 0.2757 || train_acc: 0.9089 || test_loss: 0.3814 || test_acc: 0.8813
train_loss: 0.2679 || train_acc: 0.9140 || test_loss: 0.3822 || test_acc: 0.8829
train_loss: 0.2514 || train_acc: 0.9174 || test_loss: 0.3900 || test_acc: 0.8825
train_loss: 0.2419 || train_acc: 0.9224 || test_loss: 0.3910 || test_acc: 0.8827
train_loss: 0.2347 || train_acc: 0.9238 || test_loss: 0.3784 || test_acc: 0.8853
train_loss: 0.2193 || train_